In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
data_dir = Path("data")
INDIR = Path("../data/data_raw")
OUTDIR = Path("../data/data_processed")

OUTDIR.mkdir(parents=True, exist_ok=True)

## Tratamento de PARTICIPANTES_2024

In [ ]:
arquivo = INDIR / "PARTICIPANTES_2024.csv"
df = pd.read_csv(arquivo, sep=';', encoding='latin-1')

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

display(df.head())

In [ ]:
df_participantes = df[['IN_TREINEIRO', 'SG_UF_PROVA', 'Q007', 'NO_MUNICIPIO_PROVA']]

In [ ]:
df_participantes.isnull().mean()*100

In [ ]:
df_participantes.info()

In [ ]:
df_participantes = df_participantes[df_participantes['IN_TREINEIRO'] != 1]
df_participantes = df_participantes.drop(columns=['IN_TREINEIRO'])

In [ ]:
cor_renda_map_sm = {
    "A": 0.0,
    "B": 1.0,
    "C": 1.5,
    "D": 2.0,
    "E": 2.5,
    "F": 3.0,
    "G": 4.0,
    "H": 5.0,
    "I": 6.0,
    "J": 7.0,
    "K": 8.0,
    "L": 9.0,
    "M": 10.0,
    "N": 12.0,
    "O": 15.0,
    "P": 20.0,
    "Q": 20.0
}

df_participantes['Q007'] = df_participantes['Q007'].map(cor_renda_map_sm)
df_participantes['Q007'] = df_participantes['Q007'].astype("Float64")
df_participantes = df_participantes.rename(columns={'Q007': 'RENDA_FAMILIAR_SM'})

In [ ]:
df_participantes['NO_MUNICIPIO_PROVA'] = df_participantes['NO_MUNICIPIO_PROVA'].str.upper()

In [ ]:
df_participantes.head()

In [ ]:
df_participantes.info()

In [ ]:
analise_cols = ['RENDA_FAMILIAR_SM']

for col in analise_cols:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    df_participantes[col].hist(ax=ax1, density=True)
    ax1.set_title(f'Histograma - {col}')
    ax1.set_xlabel(col)
    
    df_participantes.boxplot(column=col, ax=ax2)
    ax2.set_title(f'Box Plot - {col}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# TRANSFORMAÇÃO PARA ESCALA LOGARÍTMICA DE NET_SALES

df_participantes['RENDA_FAMILIAR_SM_LOG'] = np.log1p(df_participantes['RENDA_FAMILIAR_SM'])

analise_cols = ['RENDA_FAMILIAR_SM_LOG']

for col in analise_cols:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    df_participantes[col].hist(ax=ax1, bins=30)
    ax1.set_title(f'Histograma - {col}')
    ax1.set_xlabel(col)
    
    df_participantes.boxplot(column=col, ax=ax2)
    ax2.set_title(f'Box Plot - {col}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# TRATAMENTO DE OUTLIERS

analise_de_outliers = ['RENDA_FAMILIAR_SM_LOG']
for col in analise_de_outliers:
    Q1 = df_participantes[col].quantile(0.25)
    Q3 = df_participantes[col].quantile(0.75)
    IQR = Q3 - Q1
    df_participantes = df_participantes[(df_participantes[col] >= Q1 - 1.5*IQR) & (df_participantes[col] <= Q3 + 1.5*IQR)]

print(((df_participantes.shape[0] - df_participantes.shape[0]) / df_participantes.shape[0]) * 100)
df_participantes = df_participantes.drop(columns=['RENDA_FAMILIAR_SM_LOG'])

In [ ]:
df_municipio = df_participantes.groupby('NO_MUNICIPIO_PROVA').agg(
    RENDA_FAMILIAR_SM_MEDIA=('RENDA_FAMILIAR_SM', 'mean')
).reset_index()

In [ ]:
df_municipio.head()

## Tratamento Resultados

In [ ]:
arquivo_resultados = INDIR / "RESULTADOS_2024.csv"
df = pd.read_csv(arquivo_resultados, sep=';', encoding='latin-1')

In [ ]:
df.head()

In [ ]:
df_resultado = df[['SG_UF_PROVA', 'NO_MUNICIPIO_PROVA', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO' ]]

In [ ]:
df.head()

In [ ]:
df_resultado.isnull().mean()*100

In [ ]:
df_resultado = df_resultado[df_resultado['TP_PRESENCA_CN'] == 1]
df_resultado = df_resultado[df_resultado['TP_PRESENCA_CH'] == 1]
df_resultado = df_resultado[df_resultado['TP_PRESENCA_LC'] == 1]
df_resultado = df_resultado[df_resultado['TP_PRESENCA_MT'] == 1]

df_resultado = df_resultado.drop(columns=['TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT'])

In [ ]:
df_resultado['NO_MUNICIPIO_PROVA'] = df_resultado['NO_MUNICIPIO_PROVA'].str.upper()

In [ ]:
df_resultado.head()

In [ ]:
analise_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

for col in analise_notas:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    df_resultado[col].hist(ax=ax1, density=True)
    ax1.set_title(f'Histograma - {col}')
    ax1.set_xlabel(col)
    
    df_resultado.boxplot(column=col, ax=ax2)
    ax2.set_title(f'Box Plot - {col}')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# DESNECESSÁRIO

for col in analise_notas:
    Q1 = df_resultado[col].quantile(0.25)
    Q3 = df_resultado[col].quantile(0.75)
    IQR = Q3 - Q1
    df_resultado = df_resultado[(df_resultado[col] >= Q1 - 1.5*IQR) & (df_resultado[col] <= Q3 + 1.5*IQR)]

print(((df_resultado.shape[0] - df_resultado.shape[0]) / df_resultado.shape[0]) * 100)

In [ ]:
df_resultado.head()

In [ ]:
df_resultado = df_resultado.groupby('NO_MUNICIPIO_PROVA').agg(
    UF=('SG_UF_PROVA', 'first'),
    QTD_PARTICIPANTES=('NO_MUNICIPIO_PROVA', 'size'),
    NU_NOTA_CN_MEDIA=('NU_NOTA_CN', 'mean'),
    NU_NOTA_CH_MEDIA=('NU_NOTA_CH', 'mean'),
    NU_NOTA_LC_MEDIA=('NU_NOTA_LC', 'mean'),
    NU_NOTA_MT_MEDIA=('NU_NOTA_MT', 'mean'),
    NU_NOTA_REDACAO_MEDIA=('NU_NOTA_REDACAO', 'mean')
).reset_index()

In [ ]:
df_clustering = df_resultado.merge(
    df_municipio,
    on='NO_MUNICIPIO_PROVA',
    how='left'
)

In [ ]:
df_clustering.isnull().sum()

In [ ]:
df_clustering.head()

In [ ]:
X_scaled = df_clustering.copy()

X_scaled = X_scaled.drop(columns=['NO_MUNICIPIO_PROVA', 'RENDA_FAMILIAR_SM_MEDIA', 'UF', 'QTD_PARTICIPANTES'])

col_scatter = ['NU_NOTA_CN_MEDIA', 'NU_NOTA_CH_MEDIA', 'NU_NOTA_LC_MEDIA', 'NU_NOTA_MT_MEDIA', 'NU_NOTA_REDACAO_MEDIA']
scaler = StandardScaler()
X_scaled[col_scatter] = scaler.fit_transform(X_scaled[col_scatter])


In [ ]:
X_scaled.head()

In [ ]:
caminho_tratado = OUTDIR / "ANALISE_NOTAS_ENEM_MUNICIPIOS_BRASIL_TRATADO.csv"
df_clustering.to_csv(caminho_tratado, index=False)

caminho_modelo = OUTDIR / "ANALISE_NOTAS_ENEM_MUNICIPIOS_BRASIL_MODELO.csv"
X_scaled.to_csv(caminho_modelo, index=False)